In [1]:
import os
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=8'
from time import time
from pathlib import Path
from functools import partial

import numpy as onp
from jax import config
import jax.numpy as np
from jax.lax import integer_pow
config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
from matplotlib import colors

from gwfast.gwfastGlobals import detectors as det_dict, detPath
import gwfast.waveforms as waveforms
from gwfast.detector import Detector
import gwfast.network as network
from gwfast.signals import AGNLensedGWSignal, GeneralLensedGWSignal
from gwfast.lensing_utils import (
    compute_lensed_angles_approx,
    convert_y_from_Einstein_to_Rorbit
)
from gwfast.fisherTools import (
    reduce_Fisher_matrix,
    compute_covariance_matrix,
    covariance_change_variable,
    covariance_change_variable_1
)

/users/hin-wai.leong/src/AGN-gwfast/gwfast/waveforms.py:35: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [2]:
sensitivity_dir = Path(detPath)/'observing_scenarios_paper'
# Set up detectors
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=sensitivity_dir/'AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=sensitivity_dir/'AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=sensitivity_dir/'avirgo_O5low_NEW.txt')

wf_model = waveforms.IMRPhenomD()

H1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
HLV_AGN = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

H1_Lensed = GeneralLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
L1_Lensed = GeneralLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
V1_Lensed = GeneralLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
HLV_Lensed = network.DetNet({'H1': H1_Lensed, 'L1': L1_Lensed, 'V1': V1_Lensed})

Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8


In [3]:
from collections import OrderedDict
from gwfast.gwfastUtils import (
    noise_weighted_inner_product,
    optimal_snr,
    get_model_parameters,
    get_mass_parameters,
    apply_psi_rotation,
    ra_dec_from_th_phi_rad,
)

res = 1000
reference_parameters = {
    'Mc': 30, 'eta': 0.24, 'iota': 0.99*np.pi/2, 'phase': 2,
    'chi1z': 0.3, 'chi2z': 0.5, 'tcoal': 0,
    'R_orbit': 100, 'M_lz': 1e4, 'src_pos': 0.5,
    'dL': 1, 'psi': 4, 'theta': 1.87, 'phi': 2.66,
}
# reference_parameters['M_lz'] = 1e6
# reference_parameters['iota'] = 0.999 * np.pi / 2
reference_parameters['Mc'] = 30

jax_ref_params = {key: np.array([value]).astype(np.float64) for key, value in reference_parameters.items()}

# Need to compute the model_params once first
with_mass_params = get_mass_parameters(jax_ref_params)
fcut = H1_AGN.wf_model.fcut(**with_mass_params)
fminarr = np.full(fcut.shape, H1_AGN.fmin)
fgrids = np.geomspace(fminarr, fcut, num=int(res))

opt_snr = H1_AGN.SNRInteg(parameters=jax_ref_params)
raw_h_i = H1_AGN.signal_derivatives(
    freq_grid=fgrids, parameters=OrderedDict(jax_ref_params), computeAnalyticalDeriv=True)
h_i = OrderedDict({key: value / opt_snr for key, value in raw_h_i.items()})

In [10]:
raw_h_i['Mc']

Array([[-2.12677511e-26-4.96595594e-25j,  8.92141261e-26-1.47198526e-24j,
         2.58171911e-24-9.01639937e-25j,  2.28496414e-24+3.18411981e-24j,
        -3.22569908e-24+3.86475341e-24j, -5.50651352e-24-2.70614662e-24j,
         1.58797442e-24-7.12204390e-24j,  8.57066608e-24-2.42120548e-25j,
         2.89775138e-24+9.54738566e-24j, -9.56590325e-24+6.31814828e-24j,
        -1.01003147e-23-8.10584541e-24j,  4.88550705e-24-1.34798917e-23j,
         1.55363879e-23+1.08679204e-25j,  5.49343816e-24+1.55516941e-23j,
        -1.33184073e-23+1.08984873e-23j, -1.51895906e-23-9.18906275e-24j,
         3.81870728e-24-1.78532641e-23j,  1.87609340e-23-2.21017330e-24j,
         8.56433927e-24+1.78537447e-23j, -1.48262344e-23+1.49473273e-23j,
        -2.06419281e-23-9.17888864e-24j,  7.54584127e-25-2.42476082e-23j,
         2.40210920e-23-9.57119080e-24j,  1.96513273e-23+1.87992861e-23j,
        -8.94171822e-24+2.66808151e-23j, -2.84381788e-23+3.45146776e-24j,
        -1.53480723e-23-2.43528523e-23

# Testing psuedo-inverses and determinant

Recall that, under the AGN -> Generic transformation:
$$
T : \left({\rm BBH}, M_{\rm L}, R_{\rm orb}, \phi_{\rm L}\right)
\mapsto
\left({\rm BBH}, \delta{\cal M}_{\rm c}, \delta d_{\rm L}, \delta\iota, \delta\phi, \delta\psi, \delta t\right)
$$
Such transformation is far from being one-to-one, and it is (likely) not invertible. 

Hence, its Jacobian 
$$
J_{ij} = \frac{\partial T_i}{\partial \theta_j}\ ,
$$
would be a $m \times n$ matrix, with $m = n + 3 > n$, and is clearly singular.

The transformed Fisher matrix, $\tilde F$, is given by:
$$
\tilde F = J^T F J\ ,
$$
then by the properties of determinant, we have:
$$
\det\tilde F = (\det J)^2 (\det F)\ ,
$$
but we already had $\det J = $, hence $\det \tilde F$ must vanish as well. It follows that there will not be a well-defined inverse as well. 

But intuitively, the new Fisher matrix still carry the information of $F$, and $J$ is merely projecting that information into the new (larger) space. 
To extract this information, in the new space, we can utilise the psuedo-inverse and psuedo-determinant. 

In particular, if a (singular) matrix admits an SVD-decomposition as such:
$$
A = U \,\Sigma\,V^T\ ,
$$
where $\Sigma$ is the diagonal matrix with the singular values. Then the **psuedo-determinant** of $A$ is defined as the product of all those non-zero singular values, namely, if
$$
\Sigma = \mathop{\rm diag}\{\sigma_1, \sigma_2, \ldots, \sigma_r, 0, \ldots \} \ ,
$$
for $r$ being the rank of $A$, then its psuedo-determinant is:
$$
\vert A \vert_+ = \sigma_1 \cdot \sigma_2 \cdot \ldots \cdot \sigma_r\ .
$$
Moreover, its psuedo-inverse is defined as
$$
A^+ = V \, \Sigma^+ \, U\ ,
$$
where 
$$
\Sigma^+ = \mathop{\rm diag} \{1/\sigma_1, 1/\sigma_2, \ldots, 1/\sigma_r, 0, \ldots \}\ .
$$


In [ ]:
import mpmath
typeuse = "float128"

mp_fisher = mpmath.matrix(fisher_mat)

DEFAULT_SVD = {
    "condition_max": 1e50,
    "truncate": False,
    "svals_thresh": 1e-15,
}

def compute_single_svd_psuedo_inverse(
        mpm_fisher_mat, condition, svd_kwargs={}
        ):
    condition_max = svd_kwargs.get("condition_max", DEFAULT_SVD["condition_max"])
    truncate = svd_kwargs.get("truncate", DEFAULT_SVD["truncate"])
    svals_thresh = svd_kwargs.get("svals_thresh", DEFAULT_SVD["svals_thresh"])
    
    U, sing_vals, V = mpmath.svd_r(mpm_fisher_mat)
    S_diag = np.array(sing_vals.tolist(), dtype=typeuse)

    # Truncate singular values that differ too much
    if (truncate) and (np.abs(condition) > condition_max):
        max_sing_val = absmax(S_diag)
        S_inv = mpmath.matrix(
            np.where(
                np.abs(S_diag) / max_sing_val > svals_thresh,
                1 / S_diag,
                1 / (max_sing_val * svals_thresh)
            ).astype(typeuse)
        )

    else:
        S_inv = mpmath.matrix(1 / S_diag)

    cc = V.T * mpmath.diag(S_inv) * U.T
    return cc

In [ ]:
AGN_fisher = HLV_AGN.FisherMatr(jax_ref_params, res=100)
AGN_fisher = AGN_fisher[:, :, 0]

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.


In [13]:
sign, logdetF = np.linalg.slogdet(AGN_fisher)
print(f"log10(det F): {logdetF/np.log(10):.3f}, Sign: {sign}")

log10(det F): 31.385, Sign: 1.0


## A little fun thing to check

Define:
$$
\begin{align*}
x &= \left( {\rm BBH}, M_{\rm L}, R_{\rm orb}, y\right) \in \mathbb{R}^n \ , \\
y &= \left( {\rm BBH}, \delta {\cal M}_{\rm c}, \delta D_{\rm L}, \delta\iota, \delta\phi, \delta\psi, \delta t\right) \in \mathbb{R}^m  \ ,
\end{align*}
$$
where $m > n$. 

Then define the $m \times n$ Jacobian matrix as:
$$
J_{ij} = \frac{\partial y_i}{\partial x_j} \ .
$$

Then, we have the covariances of $y$ and $x$ are related by:
$$
\begin{equation}
C_y = J \, C_x \, J^T \ .
\end{equation}
$$

It is clear that given $J$ is singular, (not full rank), then the determinant of $C_y$ vanishes:
$$
\vert C_y\vert = \vert J \, C_x \, J^T \vert = \vert J\vert^2 \,\vert C_x\vert = 0\ .
$$

Hence, we will compute the psuedo-determinant instead:
$$
\begin{equation}
\vert C_y \vert_+ = \vert J^T J \vert \,\vert C_x\vert\ ,
\end{equation}
$$
note that in this case, $J^T J$ is an $n \times n$ matrix, a full-rank matrix, and its determinant is well-defined.

This result may not be immediate from the definition of $C_y$ and properties of determinants, after all, this is a new quantity. But such quantity would naturally fall out from the SVD decomposition of $C_y$.

So, the plan:
1. Compute $C_y$ (Eq. (1)).
2. Try to compute $\vert C_y \vert_+$ from SVD decomposition.
3. Compute $\vert C_y \vert_+$ from the relation $\vert J^T J\vert \,\vert C_x\vert$ (Eq. (2)).
4. Compare the results.

In [ ]:
def get_transform_Jacobian(
        transformation, from_params, independent_params=[] 
):
    '''Compute the Jacobian matrix of an arbitary transformation.

    :param function transformation: A function that takes in a dictionary of parameters and returns a dictionary of transformed parameters.
    :param dict from_params: The full set of parameters to be transformed, including the independent parameters.
    :param list independent_params: A list of parameter names that are considered independent. They will be excluded from the Jacobian computation.

    Returns:
        dict: A dictionary containing the Jacobian matrix of the transformation.

    Remark:
        Given that the independent parameters have shape (k,), they are not needed in the `transformation`.
        The transformation itself should be a map from (N,) to (M,).
        Then the `from_params` should have shape (k+N,), 
        The output Jacobian therefore would have shape (k+N, k+M).
    '''
    kk = len(independent_params)
    NN = len(from_params)
    MM = len(transformation(from_params))

    params_subset = {key: value for key, value in from_params.items() if key not in independent_params}
    jacobian_dict = vmap(jacfwd(transformation))(params_subset)

    output_mat = onp.zeros((kk + NN, kk + MM), dtype=typeuse)